# Projeto CardioIA - Fase 2

## Parte 1: Sistema Especialista Baseado em Regras (IA Simbólica)
**Objetivo:** Construir um "Estetoscópio Digital" básico.
Nesta etapa, criamos um mapa de conhecimento (ontologia) relacionando sintomas a doenças cardiovasculares. Utilizamos técnicas de Processamento de Linguagem Natural (NLP), como a remoção de *stopwords* e tokenização, para extrair informações de relatos de pacientes e sugerir diagnósticos automatizados com base em regras estritas.

### Etapa 1: Carregando o Mapa de Conhecimento
Nesta etapa, importamos a biblioteca nativa `csv` do Python e lemos o arquivo `mapa_conhecimento.csv`. O objetivo é transferir as regras de diagnóstico do arquivo para a memória do nosso programa (em uma lista de dicionários), permitindo que a Inteligência Artificial consulte essas regras mais à frente.


In [15]:
# 1. Importamos a biblioteca 'csv' nativa do Python para lidar com planilhas
import csv

# 2. Criamos uma lista vazia que vai guardar todas as nossas regras de diagnóstico
mapa_conhecimento = []

# 3. Abrimos o arquivo CSV em modo de leitura ('r') com a codificação correta ('utf-8')
with open('../document/mapa_conhecimento.csv', mode='r', encoding='utf-8') as arquivo_csv:

    # 4. Usamos o DictReader para ler o arquivo. Ele entende a primeira linha como cabeçalho
    leitor = csv.DictReader(arquivo_csv)

    # 5. Lemos linha por linha e guardamos na nossa lista 'mapa_conhecimento'
    for linha in leitor:
        mapa_conhecimento.append(linha)

# 6. Imprimimos uma mensagem de sucesso e verificamos se os dados foram lidos corretamente
print("Mapa de conhecimento carregado com sucesso!")
print(f"Total de regras carregadas: {len(mapa_conhecimento)}")
print("\nExemplo das 3 primeiras regras cadastradas:")
for regra in mapa_conhecimento[:3]:
    print(regra)

Mapa de conhecimento carregado com sucesso!
Total de regras carregadas: 80

Exemplo das 3 primeiras regras cadastradas:
{'Sintoma 1': 'fraqueza da face', 'Sintoma 2': 'dificuldade para falar', 'Doenca Associada': 'AVC'}
{'Sintoma 1': 'dor no peito', 'Sintoma 2': 'suor frio', 'Doenca Associada': 'Infarto'}
{'Sintoma 1': 'visão embaçada', 'Sintoma 2': 'tonturas', 'Doenca Associada': 'Hipertensão'}


### Etapa 2: Triagem Digital (Análise dos Relatos)
Nesta etapa, o sistema atua como um "Estetoscópio Digital". Vamos abrir o arquivo `frases_pacientes.txt`, ler cada relato e aplicar técnicas básicas de Processamento de Linguagem Natural (NLP). O código padroniza o texto (transformando tudo em letras minúsculas) e varre a frase em busca das combinações exatas de sintomas que definimos na nossa ontologia. Se a regra for satisfeita, a IA sugere o diagnóstico; caso contrário, o caso é classificado como inconclusivo.

In [16]:
# 1. Função inteligente de NLP para buscar sintomas ignorando palavras de ligação
def sintoma_presente(sintoma_csv, frase_paciente):
    # Stopwords: Palavras que não agregam valor médico e devem ser ignoradas
    stopwords = ['da', 'de', 'do', 'na', 'no', 'sobre', 'para', 'com', 'um', 'uma', 'e', 'o', 'a', 'em']

    # Tokenização: Quebra o sintoma em palavras e guarda só as importantes
    palavras_chave = [palavra for palavra in sintoma_csv.split() if palavra not in stopwords]

    # Verifica se TODAS as palavras-chave estão soltas dentro da frase do paciente
    for palavra in palavras_chave:
        if palavra not in frase_paciente:
            return False # Se faltar uma palavra-chave, o sintoma não está lá
    return True # Se achou todas, retorna Verdadeiro

# 2. Abrimos o arquivo de texto com as frases dos pacientes
with open('../document/frases_pacientes.txt', mode='r', encoding='utf-8') as arquivo_txt:
    frases = arquivo_txt.readlines()

print("=== INICIANDO TRIAGEM DIGITAL (CARDIO IA) ===\n")

contador_pacientes = 1

# 3. Analisamos cada frase
for frase in frases:
    frase_limpa = frase.lower().strip()

    # Ignora linhas em branco
    if frase_limpa == "":
        continue

    diagnostico_encontrado = False
    print(f"Relato do Paciente {contador_pacientes}: '{frase.strip()}'")

    # 4. O motor da IA: cruza a frase com as regras usando a nossa nova função NLP
    for regra in mapa_conhecimento:
        sintoma1 = regra['Sintoma 1'].lower()
        sintoma2 = regra['Sintoma 2'].lower()
        doenca = regra['Doenca Associada']

        # Usamos a função inteligente em vez da busca exata
        if sintoma_presente(sintoma1, frase_limpa) and sintoma_presente(sintoma2, frase_limpa):
            print(f"-> Diagnóstico Sugerido pela IA: {doenca}")
            diagnostico_encontrado = True
            break

    if not diagnostico_encontrado:
        print("-> Diagnóstico Sugerido pela IA: Inconclusivo (Encaminhar para médico humano)")

    print("-" * 60)
    contador_pacientes += 1

=== INICIANDO TRIAGEM DIGITAL (CARDIO IA) ===

Relato do Paciente 1: '"Desde a madrugada, sinto uma dor no peito muito forte que se espalha para o meu braço esquerdo, acompanhada de suor frio, o que me impediu de levantar da cama hoje."'
-> Diagnóstico Sugerido pela IA: Infarto
------------------------------------------------------------
Relato do Paciente 2: '"Há cerca de três dias venho sentindo uma dor de cabeça constante e um zumbido no ouvido irritante, atrapalhando muito minha concentração no trabalho."'
-> Diagnóstico Sugerido pela IA: Hipertensão
------------------------------------------------------------
Relato do Paciente 3: '"Hoje de manhã, percebi uma fraqueza súbita na face e dificuldade para falar com minha esposa, me deixando incapaz de tomar meu café."'
-> Diagnóstico Sugerido pela IA: AVC
------------------------------------------------------------
Relato do Paciente 4: '"Comecei a sentir uma falta de ar intensa e uma sensação de desmaio logo após o almoço, a ponto de

### Conclusão da Parte 1 e Limitações Encontradas
Ao analisarmos a saída do nosso motor de inferência, observamos um fenômeno clássico da **IA Simbólica**:
1. **O Sucesso:** O sistema com NLP foi capaz de ignorar palavras irrelevantes e diagnosticar corretamente pacientes que usaram os termos exatos mapeados na nossa base de dados.
2. **A Limitação:** O sistema falhou (retornando "Inconclusivo") quando o paciente apresentou uma combinação de sintomas válida no mundo real, mas que não estava *explicitamente* cruzada na mesma linha do nosso arquivo CSV (ex: "aperto no tórax" e "dificuldade para respirar").

**Veredito:** Sistemas baseados em regras são excelentes e rápidos, mas não possuem capacidade de generalização. Eles não "aprendem". Para resolver essa limitação e criar um sistema de triagem mais robusto, avançaremos agora para a **Parte 2**, onde utilizaremos **Machine Learning (Aprendizado de Máquina)**.

# Parte 2: Classificador Básico de Texto

Nesta etapa será desenvolvido um classificador básico de texto capaz de classificar relatos simulados de pacientes em duas categorias: baixo risco e alto risco.

Para isso, será utilizado o método TF-IDF para transformar os textos em representações numéricas e um modelo de Regressão Logística para realizar a classificação.

In [17]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [18]:
dados = pd.read_csv("../document/base_risco.csv")

print("Base de risco carregada com sucesso!")
print(f"Quantidade de registros: {len(dados)}")

print("\nPrimeiros registros:")
print(dados.head())

Base de risco carregada com sucesso!
Quantidade de registros: 20

Primeiros registros:
                                               frase    situacao
0  sinto uma dor muito forte no peito acompanhada...  alto risco
1  meu braço esquerdo adormeceu de repente e minh...  alto risco
2  estou com uma falta de ar intensa e não consig...  alto risco
3  tive uma convulsão há poucos minutos e continu...  alto risco
4  estou sangrando muito após um corte profundo n...  alto risco


In [19]:
dados = pd.read_csv("../document/base_risco.csv")

dados.columns = dados.columns.str.strip()
dados["situacao"] = dados["situacao"].str.replace('"', '', regex=False).str.strip()

print(dados.columns.tolist())
print(dados["situacao"].value_counts())

['frase', 'situacao']
situacao
alto risco     10
baixo risco    10
Name: count, dtype: int64


In [20]:
X = dados["frase"]
y = dados["situacao"]

print("Frases:")
print(X.head())

print("\nClassificações:")
print(y.head())

Frases:
0    sinto uma dor muito forte no peito acompanhada...
1    meu braço esquerdo adormeceu de repente e minh...
2    estou com uma falta de ar intensa e não consig...
3    tive uma convulsão há poucos minutos e continu...
4    estou sangrando muito após um corte profundo n...
Name: frase, dtype: str

Classificações:
0    alto risco
1    alto risco
2    alto risco
3    alto risco
4    alto risco
Name: situacao, dtype: str


In [21]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Quantidade para treino:", len(X_treino))
print("Quantidade para teste:", len(X_teste))

Quantidade para treino: 14
Quantidade para teste: 6


In [22]:
tfidf = TfidfVectorizer()

X_treino_tfidf = tfidf.fit_transform(X_treino)
X_teste_tfidf = tfidf.transform(X_teste)

print("Formato dos dados de treino:", X_treino_tfidf.shape)
print("Formato dos dados de teste:", X_teste_tfidf.shape)

Formato dos dados de treino: (14, 93)
Formato dos dados de teste: (6, 93)


In [23]:
modelo = LogisticRegression()

modelo.fit(X_treino_tfidf, y_treino)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


In [24]:
previsoes = modelo.predict(X_teste_tfidf)

print("Resultados das previsões:\n")

for frase, real, previsto in zip(X_teste, y_teste, previsoes):
    print("Frase:", frase)
    print("Classificação real:", real)
    print("Classificação prevista:", previsto)
    print("-" * 60)

Resultados das previsões:

Frase: estou sangrando muito após um corte profundo na perna
Classificação real: alto risco
Classificação prevista: baixo risco
------------------------------------------------------------
Frase: estou com uma leve cólica abdominal no final do dia
Classificação real: baixo risco
Classificação prevista: baixo risco
------------------------------------------------------------
Frase: tive uma convulsão há poucos minutos e continuo tonta
Classificação real: alto risco
Classificação prevista: baixo risco
------------------------------------------------------------
Frase: sinto uma leve azia depois de comer refeição pesada
Classificação real: baixo risco
Classificação prevista: alto risco
------------------------------------------------------------
Frase: estou com corrimento nasal e espirrando bastante devido a alergia
Classificação real: baixo risco
Classificação prevista: baixo risco
------------------------------------------------------------
Frase: meu filho i

In [25]:
acuracia = accuracy_score(y_teste, previsoes)

print("Acurácia do modelo:", acuracia)
print(f"Acurácia em porcentagem: {acuracia * 100:.2f}%")

Acurácia do modelo: 0.3333333333333333
Acurácia em porcentagem: 33.33%


In [26]:
print("Relatório de classificação:\n")

print(classification_report(
    y_teste,
    previsoes,
    zero_division=0
))

Relatório de classificação:

              precision    recall  f1-score   support

  alto risco       0.00      0.00      0.00         3
 baixo risco       0.40      0.67      0.50         3

    accuracy                           0.33         6
   macro avg       0.20      0.33      0.25         6
weighted avg       0.20      0.33      0.25         6



In [ ]:
matriz = confusion_matrix(
    y_teste,
    previsoes,
    labels=["alto risco", "baixo risco"]
)

print("Matriz de confusão:")
print(matriz)

In [ ]:
novas_frases = [
    "estou com uma dor muito forte no peito e dificuldade para respirar",
    "estou com uma leve dor de cabeça depois de estudar o dia inteiro",
    "uma pessoa desmaiou e não está respondendo",
    "estou espirrando por causa de uma alergia",
    "estou sentindo uma dor intensa no braço"
]

novas_frases_tfidf = tfidf.transform(novas_frases)

novas_previsoes = modelo.predict(novas_frases_tfidf)

print("Teste com novas frases:\n")

for frase, previsao in zip(novas_frases, novas_previsoes):
    print("Frase:", frase)
    print("Classificação prevista:", previsao)
    print("-" * 60)

## Análise dos Resultados

O modelo de Regressão Logística utilizando TF-IDF obteve uma acurácia de 33,33% no conjunto de teste. Das 6 frases utilizadas para avaliação, apenas 2 foram classificadas corretamente.

A matriz de confusão mostrou que nenhum dos 3 casos de alto risco presentes no conjunto de teste foi identificado corretamente. Além disso, um caso de baixo risco foi classificado incorretamente como alto risco.

Nos testes realizados com novas frases, o modelo apresentou resultados variados. Frases contendo expressões como "dor muito forte no peito" e "dor intensa no braço" foram classificadas como alto risco, enquanto frases relacionadas a "leve dor de cabeça" e "alergia" foram classificadas como baixo risco.

Entretanto, a frase "uma pessoa desmaiou e não está respondendo" foi classificada como baixo risco, demonstrando uma limitação importante do modelo.

O baixo desempenho pode estar relacionado principalmente ao tamanho reduzido da base de dados. Foram utilizados apenas 20 exemplos, sendo 14 destinados ao treinamento. Dessa forma, o modelo possui poucos exemplos para aprender padrões capazes de generalizar corretamente para frases diferentes.

Além disso, o TF-IDF representa os textos numericamente com base nas palavras presentes, mas não fornece ao modelo uma compreensão médica real do significado ou da gravidade dos sintomas.

Portanto, este classificador deve ser considerado apenas uma demonstração acadêmica de técnicas de processamento de texto e Machine Learning, não sendo adequado para diagnóstico ou triagem médica real.